In [1]:
# SETUP — collapsed on purpose: imports, table styling, the data load, and the title.
import cfdb_conn as cf
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

THRESHOLD = 6.0                                   # points between our line and the market's
MODEL = "cfdb_wtc_c1_own_features_tuned"          # our model (R-2490), as the warehouse names it
BREAK_EVEN = 110 / 210                            # 52.4%: the win rate needed at standard -110 odds

pd.set_option("display.max_rows", 200)
display(HTML("""
<style>
table.dataframe { border-collapse: collapse !important; border: 1px solid #8e99a6 !important;
    background-color: #ffffff !important; color: #1a1a1a !important; font-size: 12px !important; }
table.dataframe td, table.dataframe th { border: 1px solid #c3cbd5 !important;
    padding: 4px 8px !important; text-align: left !important; }
table.dataframe thead th { background-color: #2f4156 !important; color: #ffffff !important;
    font-weight: 600 !important; border-color: #2f4156 !important; position: sticky; top: 0; z-index: 2; }
table.dataframe tbody th { background-color: #eceff4 !important; color: #1a1a1a !important; }
table.dataframe tbody td { background-color: #ffffff !important; color: #1a1a1a !important; }
table.dataframe tbody tr:nth-child(even) td { background-color: #f6f8fa !important; }
table.dataframe tbody tr:hover td { background-color: #fff6d5 !important; }
</style>"""))

# One row per game our model scored: the line, our line, the gap, and the game's result if it has one.
# The notebook may join (the site may not); srv_game supplies kickoff, abbreviations and the score.
try:
    games = cf.q(f"""
        select e.season, e.split, e.week, g.start_date, e.away_team, e.home_team,
               g.away_abbreviation, g.home_abbreviation,
               e.spread, e.predicted_margin, e.edge_value, e.edge_magnitude,
               g.is_completed, g.away_points, g.home_points, e.cover_correct
        from {cf.t("srv_edge_finder")} e
        join {cf.t("srv_game")} g using (game_id)
        where e.market = 'spread' and e.model_name = :model
    """, {"model": MODEL}, limit=None)
except Exception as exc:
    print(cf._diagnose(exc) or f"{type(exc).__name__}: {exc}")
    raise

as_of = cf.freshness()["as_of_ts"].max()
display(Markdown(
    f"# Where our model and Vegas disagree by {THRESHOLD:g}+ points\n\n"
    f"Data as of {as_of:%a %b %-d, %-I:%M %p} UTC · {len(games):,} games our model has scored · rerun to update"))

# Where our model and Vegas disagree by 6+ points

Data as of Sat Oct 3, 4:00 PM UTC · 622 games our model has scored · rerun to update

**Marc's request (2026-10-03):** *"a list of all the games where my model has a spread that is at least 6 points
different than the vegas spread."*

## How to read it

Every line is the **home team's line, the market's way: negative means the home team is favored.**

| column | meaning |
|---|---|
| **Vegas line** | the market's spread (median of the books' latest lines), e.g. `UGA -25` = Georgia favored by 25 |
| **Our line** | our model's spread in the same form, e.g. `UGA -17.3` |
| **Gap** | Vegas line minus our line. **Positive: we like the home team more than Vegas does. Negative: we like the away team more** |
| **We like** | the side our line favors against Vegas |
| **Final** | the score, away – home, once the game is played |
| **Our side covered** | did "We like" beat the Vegas line? `yes` / `no` / `push` |

**Which games:** every game our model (R-2490's tuned model) has a forecast for:
- **2026, live:** forecasts written before kickoff, Week 5 onward.
- **2025, backtest:** the 2025 season scored once by a model trained only on 2016–2024. It's the track record.

In [2]:
def line(abbr, value, exact=False):
    """'UGA -25' — a line from the home team's side, in the market's sign. Vegas lines keep their own
    precision (quarter points like -1.25 exist when two books differ); ours is shown to one decimal."""
    if pd.isna(value):
        return ""
    text = f"{value:+.2f}".rstrip("0").rstrip(".") if exact else f"{value:+.1f}"
    return f"{abbr} {text}"


g = games.copy()
g["kickoff (PT)"] = (pd.to_datetime(g["start_date"], utc=True).dt.tz_convert("America/Los_Angeles")
                     .dt.strftime("%a %-m/%-d %-I:%M %p"))
g["matchup"] = g["away_team"] + " @ " + g["home_team"]
g["Vegas line"] = [line(a, v, exact=True) for a, v in zip(g["home_abbreviation"], g["spread"])]
g["Our line"] = [line(a, v) for a, v in zip(g["home_abbreviation"], g["predicted_margin"])]
g["Gap"] = g["edge_value"].round(1)
g["We like"] = np.where(g["edge_value"] > 0, g["home_team"], g["away_team"])

# Grade from the score, so a game played since the last refresh is graded too. The home team covers when
# the actual (away − home) margin comes in under the line.
margin = g["away_points"] - g["home_points"]
home_covers = np.select([margin < g["spread"], margin > g["spread"]], [True, False], default=np.nan)
played = g["is_completed"].fillna(False) & g["home_points"].notna()
we_like_home = g["edge_value"] > 0
g["Our side covered"] = np.where(~played, "", np.where(pd.isna(home_covers), "push",
                                 np.where((home_covers == 1) == we_like_home, "yes", "no")))
g["Final"] = np.where(played, g["away_points"].astype("Int64").astype(str) + " – "
                      + g["home_points"].astype("Int64").astype(str), "")

# Cross-check: where the warehouse has already graded the game, our grade must agree with it.
both = g[g["cover_correct"].notna() & g["Our side covered"].isin(["yes", "no"])]
mismatch = both[(both["Our side covered"] == "yes") != both["cover_correct"].astype(bool)]
print(f"graded by the warehouse: {len(both)} · our grade disagrees on {len(mismatch)}")
assert mismatch.empty, mismatch[["matchup", "Final", "Vegas line", "Our line", "cover_correct"]]

SHOW = ["Wk", "kickoff (PT)", "matchup", "Vegas line", "Our line", "Gap", "We like", "Final", "Our side covered"]
big = g[g["edge_magnitude"] >= THRESHOLD].rename(columns={"week": "Wk"})

graded by the warehouse: 553 · our grade disagrees on 0


## 2026 — this season's forecasts

In [3]:
live = big[big["season"] == 2026].sort_values(["Wk", "start_date", "edge_magnitude"],
                                             ascending=[True, True, False])
n_live = int((g["season"] == 2026).sum())
display(Markdown(f"**{len(live)} of {n_live} games** have a gap of {THRESHOLD:g} points or more."))
display(live[SHOW].reset_index(drop=True) if len(live) else Markdown("*none*"))

**9 of 56 games** have a gap of 6 points or more.

,Wk,kickoff (PT),matchup,Vegas line,Our line,Gap,We like,Final,Our side covered
0,5,Thu 10/1 6:00 PM,North Texas @ Tulsa,TLSA -1.25,TLSA +7.3,-8.6,North Texas,45 – 44,yes
1,5,Fri 10/2 4:00 PM,Pittsburgh @ Virginia Tech,VT -3,VT +4.9,-7.9,Pittsburgh,35 – 33,yes
2,5,Sat 10/3 8:00 AM,Memphis @ Charlotte,CLT +20.5,CLT +13.7,6.8,Charlotte,,
3,5,Sat 10/3 9:00 AM,Syracuse @ UConn,CONN +6.5,CONN -7.8,14.3,UConn,,
4,5,Sat 10/3 9:45 AM,Vanderbilt @ Georgia,UGA -24,UGA -17.3,-6.7,Vanderbilt,,
5,5,Sat 10/3 12:30 PM,Old Dominion @ Georgia State,GAST -2.25,GAST -10.4,8.1,Georgia State,,
6,5,Sat 10/3 12:30 PM,Ohio State @ Iowa,IOWA +14.25,IOWA +7.4,6.8,Iowa,,
7,5,Sat 10/3 12:45 PM,Marshall @ James Madison,JMU -18.5,JMU -27.4,8.9,James Madison,,
8,5,Sat 10/3 7:30 PM,Baylor @ Arizona State,ASU -4,ASU +2.4,-6.4,Baylor,,


## 2025 — the track record

The same rule on last season, from a model that had never seen it. **Small sample — read the range, not just the
rate:** a few dozen games can land well above or below 50% by luck alone.

In [4]:
hist = big[big["season"] == 2025].sort_values(["Wk", "start_date"])
done = hist[hist["Our side covered"].isin(["yes", "no"])]
hits, misses = int((done["Our side covered"] == "yes").sum()), int((done["Our side covered"] == "no").sum())
pushes = int((hist["Our side covered"] == "push").sum())
rate = hits / (hits + misses) if hits + misses else float("nan")
half = 1.96 * np.sqrt(rate * (1 - rate) / (hits + misses)) if hits + misses else float("nan")
display(Markdown(
    f"**{len(hist)} of {int((g['season'] == 2025).sum())} games** had a gap of {THRESHOLD:g}+. "
    f"Our side covered **{hits}–{misses}–{pushes}** (covered–missed–push): **{rate:.1%}**, "
    f"95% range {rate - half:.0%}–{rate + half:.0%}. Break-even at standard odds is {BREAK_EVEN:.1%}."))
display(hist[SHOW].reset_index(drop=True))

**34 of 566 games** had a gap of 6+. Our side covered **13–20–1** (covered–missed–push): **39.4%**, 95% range 23%–56%. Break-even at standard odds is 52.4%.

,Wk,kickoff (PT),matchup,Vegas line,Our line,Gap,We like,Final,Our side covered
0,5,Sat 9/27 9:00 AM,Rutgers @ Minnesota,MINN -3.5,MINN -12.2,8.7,Minnesota,28 – 31,no
1,5,Sat 9/27 9:00 AM,Bowling Green @ Ohio,OHIO -7.5,OHIO +0.0,-7.5,Bowling Green,20 – 35,no
2,5,Sat 9/27 1:00 PM,Tulane @ Tulsa,TLSA +14.5,TLSA +8.3,6.2,Tulsa,31 – 14,no
3,5,Sat 9/27 4:30 PM,App State @ Boise State,BOIS -16.5,BOIS -10.2,-6.3,App State,14 – 47,no
4,5,Sat 9/27 4:30 PM,Oregon @ Penn State,PSU -4,PSU +2.3,-6.3,Oregon,30 – 24,yes
5,5,Sat 9/27 4:30 PM,Washington State @ Colorado State,CSU -4.5,CSU -13.5,9.0,Colorado State,20 – 3,no
6,6,Sat 10/4 9:00 AM,Clemson @ North Carolina,UNC +15.5,UNC +9.4,6.1,North Carolina,38 – 10,no
7,6,Sat 10/4 11:30 AM,Western Michigan @ Massachusetts,MASS +12.5,MASS +6.2,6.3,Massachusetts,21 – 3,no
8,6,Sat 10/4 12:30 PM,UL Monroe @ Northwestern,NU -12.5,NU -5.2,-7.3,UL Monroe,7 – 42,no
9,6,Sat 10/4 4:00 PM,South Alabama @ Troy,TROY +2.5,TROY -3.5,6.0,Troy,24 – 31,yes
